# Trening MobileNetV3 z transfer learning na RAF-DB (Google Colab)

Notebook trenuje klasyfikator emocji na bazie **MobileNetV3-Large** pretrenowanej na ImageNet, fine-tuningowanej na **RAF-DB** (7 klas emocji).

**Czas treningu:** ~15-25 min na 20 epok (T4 GPU).

## Strategia transfer learning
1. **Faza 1 (warm-up, 3 epoki):** backbone zamrozony, trening tylko nowej glowy klasyfikatora (LR=1e-3)
2. **Faza 2 (fine-tuning):** odmrozony caly model, niski LR (1e-4) z cosine annealing

## Kolejnosc krokow
1. Wlacz GPU: `Runtime` -> `Change runtime type` -> `T4 GPU`
2. Uruchom komorki po kolei
3. Pobierz `mobilenetv3_best.pth` + wykresy + raport na koniec

## 1. Sprawdzenie GPU

In [ ]:
import torch
print('CUDA dostepne:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'brak')
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Dataset RAF-DB

Struktura oczekiwana przez `train_mobilenetv3.py`:
```
RAF-DB/DATASET/
  train/
    1/  2/  3/  4/  5/  6/  7/    # foldery klas (Surprise..Neutral)
  test/
    1/  2/  3/  4/  5/  6/  7/
```

**Opcja A** - mount Google Drive (jezeli wgrales `RAF-DB.zip` do swojego Drive):

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ZMIEN sciezke na swoja lokalizacje archiwum RAF-DB.zip
!cp '/content/drive/MyDrive/RAF-DB.zip' /content/RAF-DB.zip
!unzip -q /content/RAF-DB.zip -d /content/
!ls /content/RAF-DB/DATASET

**Opcja B** - pobranie z Kaggle (wymaga `kaggle.json` z Twojego konta Kaggle):

In [ ]:
# Wgraj kaggle.json przez panel po lewej (Files), potem odkomentuj:
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !pip install -q kaggle
# !kaggle datasets download -d shuvoalok/raf-db-dataset -p /content/
# !unzip -q /content/raf-db-dataset.zip -d /content/RAF-DB
# !ls /content/RAF-DB/DATASET

## 3. Kod treningu

Wgraj `train_mobilenetv3.py` recznie przez panel Files (ikona folderu po lewej) ALBO wklej zawartosc do komorki ponizej (`%%writefile train_mobilenetv3.py`).

Po wgraniu sprawdz:

In [ ]:
!ls
# Powinno byc widac: train_mobilenetv3.py, RAF-DB/ (lub /content/RAF-DB/)

## 4. Trening

Domyslnie: 20 epok lacznie (3 warm-up + 17 fine-tuning), batch 64, augmentacja, class weights, cosine annealing, early stopping (patience=5).

In [ ]:
!python train_mobilenetv3.py \
    --data_dir /content/RAF-DB/DATASET \
    --output_dir /content/out \
    --epochs 20 \
    --warmup_epochs 3 \
    --batch_size 64 \
    --num_workers 2

## 5. Podglad wynikow

In [ ]:
from IPython.display import Image, display
display(Image('/content/out/learning_curves.png'))
display(Image('/content/out/confusion_matrix.png'))

with open('/content/out/classification_report.txt') as f:
    print(f.read())

## 6. Pobranie wynikow

In [ ]:
# Spakuj wszystko i sciagnij jeden ZIP
!cd /content/out && zip -r /content/results_mobilenetv3.zip . && ls -la /content/results_mobilenetv3.zip
from google.colab import files
files.download('/content/results_mobilenetv3.zip')

**Po pobraniu rozpakuj** - znajdziesz tam:
- `mobilenetv3_best.pth` - wagi najlepszego modelu (do uzycia w detect.py / webapp)
- `learning_curves.png` - krzywe uczenia (loss + accuracy, z zaznaczona granica warm-up/fine-tuning)
- `confusion_matrix.png` - confusion matrix na tescie
- `classification_report.txt` - precision/recall/f1 dla kazdej klasy
- `history.json` - pelna historia treningu
- `config.json` - konfiguracja eksperymentu (hiperparametry + final accuracy)

## 7. Eksperyment porownawczy - bez augmentacji (opcjonalnie)

Do sprawozdania warto pokazac wplyw augmentacji na transfer learning. Drugi run bez augmentacji:

In [ ]:
!python train_mobilenetv3.py \
    --data_dir /content/RAF-DB/DATASET \
    --output_dir /content/out_no_aug \
    --epochs 20 \
    --warmup_epochs 3 \
    --batch_size 64 \
    --num_workers 2 \
    --no_augment

## 8. Eksperyment porownawczy - MobileNetV3-Small (opcjonalnie)

Mniejszy wariant (~2.5M parametrow vs ~5.5M w Large) - pokazuje trade-off rozmiar/dokladnosc:

In [ ]:
!python train_mobilenetv3.py \
    --data_dir /content/RAF-DB/DATASET \
    --output_dir /content/out_small \
    --variant small \
    --epochs 20 \
    --warmup_epochs 3 \
    --batch_size 64 \
    --num_workers 2